# 마스킹 전후 생성 품질 평가

이 노트북은 Hugging Face에 올린 로컬 마스킹 모델을 먼저 로드해보고, 같은 목업 채용 데이터를 두 체인으로 실행해 최종 산출물 품질을 비교합니다.

비교 대상은 다음 두 가지입니다.

- **일반 체인**: 마스킹 없이 회사/JD/이력서를 사용합니다. 체크리스트는 회사/JD와 DB 검색 결과로 생성하고, 분석 그래프 내부에서 STAR 분석을 수행한 뒤 최종 리포트와 면접 질문지를 생성합니다.
- **마스킹 체인**: 로컬 HF 모델로 회사/JD/이력서를 마스킹합니다. 마스킹된 회사/JD와 DB 검색 결과로 체크리스트를 생성하고, 분석 그래프 내부에서 STAR 분석을 수행한 뒤 최종 리포트와 면접 질문지를 생성합니다. 마지막에 `unmask()`로 결과물을 복호화합니다.

마지막에는 LLM judge가 두 결과를 원본 입력 기준으로 평가합니다. 평가 지표는 핵심 내용 보존, 중요 정보 누락, 환각, 체크리스트 반영, 리포트 품질, 질문지 품질, 복호화 품질입니다.

> 실행 순서: 반드시 첫 번째 코드 셀에서 HF 모델 로드 확인을 먼저 통과시킨 뒤 아래 셀들을 실행하세요.

In [1]:
from __future__ import annotations

import contextlib
import importlib.util
import inspect
import json
import os
import re
import sys
import types
from pathlib import Path
from typing import Any, TypedDict


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "backend" / "common").exists():
            return candidate
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다. 노트북을 프로젝트 내부에서 실행하세요.")


PROJECT_ROOT = find_project_root()
BACKEND_DIR = PROJECT_ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

try:
    from common.utils import load_env

    load_env()
except Exception:
    try:
        from dotenv import load_dotenv

        load_dotenv(BACKEND_DIR / ".env", encoding="utf-8")
    except Exception:
        pass


BASE_MODEL_NAME = os.getenv("MASKING_BASE_MODEL", "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct")
ADAPTER_MODEL_NAME = os.getenv("MASKING_ADAPTER_MODEL", "dlfp22/exaone-masking-lora-best")
HF_TOKEN = os.getenv("HF_TOKEN")
USE_4BIT = os.getenv("MASKING_USE_4BIT", "1").lower() not in {"0", "false", "no"}
HF_SMOKE_TEST = os.getenv("MASKING_QUALITY_HF_SMOKE_TEST", "1").lower() not in {"0", "false", "no"}
SMOKE_MAX_NEW_TOKENS = int(os.getenv("MASKING_QUALITY_SMOKE_MAX_NEW_TOKENS", "128"))
MASKING_MAX_NEW_TOKENS = int(os.getenv("MASKING_MAX_NEW_TOKENS", "512"))

LOCAL_HF_REQUIRED_PACKAGES = ["torch", "transformers", "peft", "accelerate", "sentencepiece", "safetensors"]


def missing_local_hf_packages() -> list[str]:
    return [
        package
        for package in LOCAL_HF_REQUIRED_PACKAGES
        if importlib.util.find_spec(package) is None
    ]


missing_packages = missing_local_hf_packages()
if missing_packages:
    raise ModuleNotFoundError(
        "로컬 Hugging Face 마스킹 모델 실행에 필요한 패키지가 없습니다: "
        + ", ".join(missing_packages)
        + "\n설치 예시: %pip install torch transformers peft accelerate sentencepiece safetensors"
    )

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def should_use_4bit() -> bool:
    return USE_4BIT and torch.cuda.is_available() and importlib.util.find_spec("bitsandbytes") is not None


def _patch_transformers_compat() -> None:
    try:
        import transformers.utils.generic as generic_utils

        if not hasattr(generic_utils, "maybe_autocast"):
            generic_utils.maybe_autocast = lambda *args, **kwargs: contextlib.nullcontext()
    except Exception as exc:
        print(f"[WARN] maybe_autocast patch skipped: {exc}")

    try:
        import transformers.modeling_rope_utils as rope_utils

        if not hasattr(rope_utils, "RopeParameters"):
            class RopeParameters(TypedDict, total=False):
                rope_type: str
                factor: float
                low_freq_factor: float
                high_freq_factor: float
                original_max_position_embeddings: int
                attention_factor: float
                beta_fast: float
                beta_slow: float
                short_factor: list[float]
                long_factor: list[float]

            rope_utils.RopeParameters = RopeParameters
    except Exception as exc:
        print(f"[WARN] RopeParameters patch skipped: {exc}")

    try:
        import transformers.integrations as tf_integrations

        def noop_kernel_patch(*args: Any, **kwargs: Any):
            if args and callable(args[0]) and len(args) == 1:
                return args[0]

            def decorator(fn):
                return fn

            return decorator

        for name in ("use_kernel_forward_from_hub", "use_kernel_func_from_hub", "use_kernelized_func"):
            if not hasattr(tf_integrations, name):
                setattr(tf_integrations, name, noop_kernel_patch)
    except Exception as exc:
        print(f"[WARN] kernel integration patch skipped: {exc}")


def _patch_all_create_causal_mask_refs() -> None:
    def make_compat(original_func):
        if getattr(original_func, "_exaone_input_embeds_compat", False):
            return original_func

        params = inspect.signature(original_func).parameters

        def compat(*args: Any, **kwargs: Any):
            if "input_embeds" in kwargs and "input_embeds" not in params:
                value = kwargs.pop("input_embeds")
                if "inputs_embeds" in params:
                    kwargs["inputs_embeds"] = value
                elif "input_tensor" in params:
                    kwargs["input_tensor"] = value
                else:
                    kwargs["input_ids"] = value
            elif "inputs_embeds" in kwargs and "inputs_embeds" not in params:
                value = kwargs.pop("inputs_embeds")
                if "input_embeds" in params:
                    kwargs["input_embeds"] = value
                elif "input_tensor" in params:
                    kwargs["input_tensor"] = value
                else:
                    kwargs["input_ids"] = value

            accepts_var_kwargs = any(param.kind == inspect.Parameter.VAR_KEYWORD for param in params.values())
            if not accepts_var_kwargs:
                kwargs = {key: value for key, value in kwargs.items() if key in params}
            return original_func(*args, **kwargs)

        compat._exaone_input_embeds_compat = True
        return compat

    patched_count = 0
    for module in list(sys.modules.values()):
        if module is None or not hasattr(module, "create_causal_mask"):
            continue
        original_func = getattr(module, "create_causal_mask", None)
        if not callable(original_func):
            continue
        try:
            compat_func = make_compat(original_func)
        except (TypeError, ValueError):
            continue
        if compat_func is not original_func:
            setattr(module, "create_causal_mask", compat_func)
            patched_count += 1
    print(f"[PATCH] create_causal_mask 호환 패치 적용 모듈 수: {patched_count}")


def _patch_exaone_model(model):
    if getattr(model, "_exaone_compat_patched", False):
        return model

    if hasattr(model, "transformer") and hasattr(model.transformer, "wte"):
        embed = model.transformer.wte
    elif hasattr(model, "transformer") and hasattr(model.transformer, "embed_tokens"):
        embed = model.transformer.embed_tokens
    elif hasattr(model, "model") and hasattr(model.model, "embed_tokens"):
        embed = model.model.embed_tokens
    else:
        embed = None

    if embed is not None:
        model.get_input_embeddings = lambda: embed
        model.set_input_embeddings = lambda value: setattr(embed, "weight", value.weight)

    _patch_all_create_causal_mask_refs()

    backbone = getattr(model, "transformer", None) or getattr(model, "model", None)
    if backbone is not None and not hasattr(backbone, "_exaone_forward_patched"):
        original_forward = backbone.forward

        def patched_forward(self, *args: Any, **kwargs: Any):
            if "input_embeds" in kwargs and "inputs_embeds" not in kwargs:
                kwargs["inputs_embeds"] = kwargs.pop("input_embeds")
            return original_forward(*args, **kwargs)

        backbone.forward = types.MethodType(patched_forward, backbone)
        backbone._exaone_forward_patched = True

    model._exaone_compat_patched = True
    return model


_masking_tokenizer = None
_masking_model = None


def get_local_masking_model():
    global _masking_tokenizer, _masking_model
    if _masking_tokenizer is not None and _masking_model is not None:
        return _masking_tokenizer, _masking_model

    _patch_transformers_compat()

    tokenizer = AutoTokenizer.from_pretrained(
        ADAPTER_MODEL_NAME,
        token=HF_TOKEN,
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model_kwargs: dict[str, Any] = {
        "token": HF_TOKEN,
        "trust_remote_code": True,
    }
    if torch.cuda.is_available():
        model_kwargs["device_map"] = "auto"
        if should_use_4bit():
            compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            model_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=compute_dtype,
                bnb_4bit_use_double_quant=True,
            )
        else:
            model_kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    print(f"BASE_MODEL_NAME={BASE_MODEL_NAME}")
    print(f"ADAPTER_MODEL_NAME={ADAPTER_MODEL_NAME}")
    print(f"CUDA available={torch.cuda.is_available()}, use_4bit={should_use_4bit()}")

    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, **model_kwargs)
    base_model = _patch_exaone_model(base_model)
    model = PeftModel.from_pretrained(base_model, ADAPTER_MODEL_NAME, token=HF_TOKEN)
    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = True

    _patch_all_create_causal_mask_refs()
    _masking_tokenizer = tokenizer
    _masking_model = model
    return tokenizer, model


def parse_masking_json(text: str) -> dict[str, list[str]]:
    label_cats = [
        "comp_name",
        "person_name",
        "address",
        "personal_info",
        "school_edu",
        "project_name",
        "jd_discrimination",
    ]
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    first = cleaned.find("{")
    last = cleaned.rfind("}")
    if first == -1 or last == -1 or first >= last:
        raise ValueError(f"마스킹 모델 출력에서 JSON object를 찾지 못했습니다: {cleaned[:300]}")
    obj = json.loads(cleaned[first : last + 1])
    return {
        key: [str(value) for value in obj.get(key, []) if str(value).strip()]
        if isinstance(obj.get(key, []), list)
        else []
        for key in label_cats
    }


MASKING_SYSTEM_PROMPT = """
당신은 한국어 채용 데이터의 개인정보 및 민감 표현 마스킹 전문가입니다.
입력 JSON에서 마스킹이 필요한 원문 표현을 찾아 아래 7개 카테고리로 분류해 JSON object 하나만 반환하세요.

카테고리:
- comp_name: 회사명, 기관명, 고객사명, 이전 근무처명, 조직 식별명
- person_name: 지원자 본인, 교수, 추천인, 동료 등 사람 이름
- address: 주소, 출신지, 거주지
- personal_info: 연락처, 고유식별정보, 생년월일, 나이, 성별, 병역, 장애, 가족, 종교, 정치성향 등 민감 정보
- school_edu: 학교명, 교육기관명, 부트캠프명
- project_name: 내부 프로젝트명, 고객사 식별 가능 프로젝트명
- jd_discrimination: JD의 차별 소지 표현

반드시 아래 7개 키만 포함하는 JSON object 하나만 출력하세요.
{
  "comp_name": [],
  "person_name": [],
  "address": [],
  "personal_info": [],
  "school_edu": [],
  "project_name": [],
  "jd_discrimination": []
}
""".strip()


def predict_masking(input_text: str, max_new_tokens: int = MASKING_MAX_NEW_TOKENS) -> str:
    tokenizer, model = get_local_masking_model()
    messages = [
        {"role": "system", "content": MASKING_SYSTEM_PROMPT},
        {"role": "user", "content": input_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = inputs.to(next(model.parameters()).device)

    _patch_all_create_causal_mask_refs()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()


def invoke_local_hf_masking(data: dict[str, Any]) -> dict[str, Any]:
    input_text = json.dumps(data, ensure_ascii=False, indent=2)
    raw = predict_masking(input_text)
    return {"raw": raw, "result": parse_masking_json(raw)}


tokenizer, masking_model = get_local_masking_model()
print("HF 마스킹 모델 로드 성공")

if HF_SMOKE_TEST:
    smoke_input = {"resume": {"name": "홍길동", "school": "한국대학교", "company": "샘플테크"}}
    smoke_raw = predict_masking(json.dumps(smoke_input, ensure_ascii=False), max_new_tokens=SMOKE_MAX_NEW_TOKENS)
    print("HF 마스킹 모델 smoke test raw output:")
    print(smoke_raw)
    print("HF 마스킹 모델 smoke test parsed output:")
    print(parse_masking_json(smoke_raw))

c:\Users\usre\miniconda3\envs\web_service_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASE_MODEL_NAME=LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct
ADAPTER_MODEL_NAME=dlfp22/exaone-masking-lora-best
CUDA available=False, use_4bit=False


[transformers] The `check_model_inputs` decorator is deprecated in favor of `merge_with_config_defaults`.


[ERROR] `cache_position` is part of ExaoneModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in C:\Users\usre\.cache\huggingface\modules\transformers_modules\LGAI_hyphen_EXAONE\EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct\ccce25bd39c141fe053e0bc75818a8f5fe962802\modeling_exaone.py.
[ERROR] `cache_position` is part of ExaoneForCausalLM.forward's signature, but not documented. Make sure to add it to the docstring of the function in C:\Users\usre\.cache\huggingface\modules\transformers_modules\LGAI_hyphen_EXAONE\EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct\ccce25bd39c141fe053e0bc75818a8f5fe962802\modeling_exaone.py.


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 2051.51it/s]


ModuleNotFoundError: No module named 'torchvision'

## 데이터와 프로젝트 체인 준비

기본 입력 파일은 `backend/common/eval/recruiting_mock_dataset.csv`입니다. 이 파일은 `company information`, `job_description`, `resume` 세 컬럼을 가진 JSON 문자열 CSV입니다.

`MASKING_QUALITY_SAMPLE_SIZE` 환경변수로 실행 샘플 수를 조절할 수 있습니다. 기본값은 3개입니다.

In [ ]:
import csv
import time
from copy import deepcopy

try:
    import pandas as pd
except ImportError:
    pd = None

from IPython.display import Markdown, display

from common import analysis_graph, checklist_agent
from common.utils import mask, unmask


EVAL_DIR = BACKEND_DIR / "common" / "eval"
DATA_PATH = EVAL_DIR / "recruiting_mock_dataset.csv"
CHAIN_CACHE_PATH = EVAL_DIR / "masking_quality_chain_outputs.json"
JUDGE_CACHE_PATH = EVAL_DIR / "masking_quality_judge_results.json"

COMPANY_COL = "company information"
JD_COL = "job_description"
RESUME_COL = "resume"
SAMPLE_SIZE = int(os.getenv("MASKING_QUALITY_SAMPLE_SIZE", "3"))
CHECKLIST_COUNT = int(os.getenv("MASKING_QUALITY_CHECKLIST_COUNT", "10"))
FORCE_REGENERATE = os.getenv("MASKING_QUALITY_FORCE_REGENERATE", "0").lower() in {"1", "true", "yes"}
FORCE_REJUDGE = os.getenv("MASKING_QUALITY_FORCE_REJUDGE", "0").lower() in {"1", "true", "yes"}


def read_csv_rows(path: Path) -> list[dict[str, Any]]:
    with path.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def parse_json_cell(value: Any, field_name: str) -> Any:
    if isinstance(value, (dict, list)):
        return value
    text = str(value or "").strip()
    if not text:
        raise ValueError(f"{field_name} 값이 비어 있습니다.")
    return json.loads(text)


def row_to_payload(row: dict[str, Any], index: int) -> dict[str, Any]:
    return {
        "set_id": int(row.get("set_id") or index),
        "company": parse_json_cell(row[COMPANY_COL], COMPANY_COL),
        "jd": parse_json_cell(row[JD_COL], JD_COL),
        "resume": parse_json_cell(row[RESUME_COL], RESUME_COL),
    }


rows = read_csv_rows(DATA_PATH)
samples = [row_to_payload(row, index) for index, row in enumerate(rows[:SAMPLE_SIZE])]

preview_rows = [
    {
        "set_id": item["set_id"],
        "company_name": item["company"].get("company_name", ""),
        "job_name": item["jd"].get("job_name", ""),
        "resume_name": item["resume"].get("name", ""),
    }
    for item in samples
]

if pd is not None:
    display(pd.DataFrame(preview_rows))
else:
    display(preview_rows)

## 체인 실행 함수

체크리스트 생성은 두 체인 모두 회사/JD 기반 query 생성, Pinecone DB 검색, 체크리스트 생성 순서로 수행합니다.

- 일반 체인: 원본 회사/JD로 체크리스트 생성 후 원본 입력으로 분석 그래프 실행
- 마스킹 체인: 로컬 HF 마스킹 결과를 적용한 회사/JD로 체크리스트 생성 후 마스킹 입력으로 분석 그래프 실행, 마지막에 복호화

In [ ]:
def save_json(path: Path, data: Any) -> None:
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def load_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def split_analysis_result(result: dict[str, Any]) -> dict[str, Any]:
    questions = result.get("question") or result.get("questions") or []
    report = {key: value for key, value in result.items() if key not in {"question", "questions"}}
    return {"report": report, "questions": questions, "full_result": result}


def generate_checklist(company: dict[str, Any], jd: dict[str, Any], mask_result: dict[str, Any] | None = None) -> dict[str, Any]:
    query = checklist_agent.invoke_extract_query_node(compinfo=deepcopy(company), jdinfo=deepcopy(jd))
    db_data = checklist_agent.invoke_search_embedding_node(query=query, cnt=CHECKLIST_COUNT)
    prompt_db_data = db_data
    if mask_result:
        prompt_db_data = mask({"db_data": deepcopy(db_data)}, mask_result)["db_data"]

    checklist = checklist_agent.invoke_fit_checklist_node(
        company_info=deepcopy(company),
        jd_info=deepcopy(jd),
        db_data=deepcopy(prompt_db_data),
        checklist_count=CHECKLIST_COUNT,
    )
    return {
        "query": query,
        "db_data": db_data,
        "prompt_db_data": prompt_db_data,
        "checklist": checklist,
    }


def run_no_mask_chain(payload: dict[str, Any]) -> dict[str, Any]:
    checklist_bundle = generate_checklist(payload["company"], payload["jd"])
    analysis_result = analysis_graph.invoke(
        company_dict=deepcopy(payload["company"]),
        jd_dict=deepcopy(payload["jd"]),
        checklist=deepcopy(checklist_bundle["checklist"]),
        resume_dict=deepcopy(payload["resume"]),
    )
    result = split_analysis_result(analysis_result)
    result["checklist_generation"] = checklist_bundle
    return result


def run_masked_chain(payload: dict[str, Any]) -> dict[str, Any]:
    source_input = {
        "company": deepcopy(payload["company"]),
        "jd": deepcopy(payload["jd"]),
        "resume": deepcopy(payload["resume"]),
    }
    masking_output = invoke_local_hf_masking(source_input)
    mask_result = masking_output["result"]
    masked_input = mask(deepcopy(source_input), mask_result)
    checklist_bundle = generate_checklist(masked_input["company"], masked_input["jd"], mask_result=mask_result)

    masked_analysis_result = analysis_graph.invoke(
        company_dict=deepcopy(masked_input["company"]),
        jd_dict=deepcopy(masked_input["jd"]),
        checklist=deepcopy(checklist_bundle["checklist"]),
        resume_dict=deepcopy(masked_input["resume"]),
    )
    unmasked_analysis_result = unmask(deepcopy(masked_analysis_result), mask_result)
    result = split_analysis_result(unmasked_analysis_result)
    result["mask_result"] = mask_result
    result["masking_raw"] = masking_output["raw"]
    result["masked_input"] = masked_input
    result["masked_raw_result"] = masked_analysis_result
    result["checklist_generation"] = checklist_bundle
    result["unmasked_checklist_generation"] = unmask(deepcopy(checklist_bundle), mask_result)
    return result

## 일반 체인과 마스킹 체인 실행

이 셀은 OpenAI, Pinecone, 로컬 HF 모델을 모두 사용하므로 시간이 걸릴 수 있습니다. 이미 실행한 결과가 있으면 캐시를 사용합니다. 다시 실행하려면 `MASKING_QUALITY_FORCE_REGENERATE=1`을 설정하세요.

In [ ]:
if CHAIN_CACHE_PATH.exists() and not FORCE_REGENERATE:
    chain_records = load_json(CHAIN_CACHE_PATH)
    print(f"캐시 로드: {CHAIN_CACHE_PATH} ({len(chain_records)}건)")
else:
    chain_records = []
    for payload in samples:
        sid = payload["set_id"]
        started_at = time.time()
        print(f"[set_id={sid}] 일반 체인 실행 시작")
        no_mask_output = run_no_mask_chain(payload)
        print(f"[set_id={sid}] 마스킹 체인 실행 시작")
        masked_output = run_masked_chain(payload)
        elapsed_sec = round(time.time() - started_at, 2)

        chain_records.append(
            {
                "set_id": sid,
                "source": {
                    "company": payload["company"],
                    "jd": payload["jd"],
                    "resume": payload["resume"],
                },
                "no_mask": no_mask_output,
                "masked": masked_output,
                "elapsed_sec": elapsed_sec,
            }
        )
        save_json(CHAIN_CACHE_PATH, chain_records)
        print(f"[set_id={sid}] 완료: {elapsed_sec}초")

print(f"총 {len(chain_records)}건 준비 완료")

## 결과물 순서대로 확인

각 샘플마다 일반 체인 결과를 먼저 보고, 그 다음 마스킹 체인에서 생성 후 복호화한 결과를 봅니다. 결과물은 최종 리포트와 면접 질문지입니다.

In [ ]:
def find_mask_tokens(obj: Any) -> list[str]:
    text = json.dumps(obj, ensure_ascii=False)
    return sorted(set(re.findall(r"\[[A-Z_]+_\d+\]", text)))


def show_questions(questions: list[dict[str, Any]]) -> None:
    if pd is not None:
        display(pd.DataFrame(questions))
    else:
        display(questions)


def display_chain_output(record: dict[str, Any]) -> None:
    sid = record["set_id"]
    display(Markdown(f"## set_id={sid}"))

    display(Markdown("### 1. 일반 체인 결과: 마스킹 없음, STAR 분석 있음"))
    display(Markdown("#### 생성 체크리스트"))
    display(record["no_mask"]["checklist_generation"]["checklist"])
    display(Markdown("#### 최종 리포트"))
    display(record["no_mask"]["report"])
    display(Markdown("#### 면접 질문지"))
    show_questions(record["no_mask"]["questions"])

    display(Markdown("### 2. 마스킹 체인 결과: 마스킹 적용, STAR 분석 있음, 최종 복호화"))
    display(Markdown("#### 마스킹 결과"))
    display(record["masked"]["mask_result"])
    display(Markdown("#### 생성 체크리스트: 복호화 후 표시"))
    display(record["masked"]["unmasked_checklist_generation"]["checklist"])
    display(Markdown(f"잔여 마스킹 토큰: `{find_mask_tokens(record['masked']['full_result'])}`"))
    display(Markdown("#### 최종 리포트"))
    display(record["masked"]["report"])
    display(Markdown("#### 면접 질문지"))
    show_questions(record["masked"]["questions"])


for record in chain_records:
    display_chain_output(record)

## LLM judge 루브릭

LLM judge는 원본 회사/JD/이력서를 기준으로 일반 체인과 마스킹 체인의 최종 산출물을 각각 독립 채점합니다. 모든 점수는 1~5점입니다.

- **source_fidelity**: 원본 근거를 왜곡하지 않고 보존했는가
- **critical_information_retention**: 채용 판단에 중요한 회사명, 직무, 기술, 경력, 학력, 자기소개서 근거가 빠지지 않았는가
- **coverage**: 체크리스트, 직무 요구사항, 지원자 핵심 경험을 충분히 다뤘는가
- **hallucination_control**: 원본에 없는 경험, 성과, 수치, 회사 내부 상황을 만들지 않았는가
- **report_quality**: 최종 리포트가 일관적이고 채용 검토에 쓸 수 있는가
- **question_quality**: 질문, 모범 답안, 질문 의도가 근거 기반이고 면접 검증에 유용한가
- **entity_recovery**: 마스킹 체인의 복호화 결과가 자연스럽고 잔여 마스킹 토큰이 없는가

`total_score`는 위 7개 지표 평균입니다. 마지막 비교표에서는 `masked - no_mask` 차이를 계산합니다.

In [ ]:
from typing import Literal

from openai import OpenAI
from pydantic import BaseModel, Field


JUDGE_MODEL = os.getenv("MASKING_QUALITY_JUDGE_MODEL", "gpt-4o-mini")


class ChainQualityScore(BaseModel):
    source_fidelity: int = Field(ge=1, le=5)
    critical_information_retention: int = Field(ge=1, le=5)
    coverage: int = Field(ge=1, le=5)
    hallucination_control: int = Field(ge=1, le=5)
    report_quality: int = Field(ge=1, le=5)
    question_quality: int = Field(ge=1, le=5)
    entity_recovery: int = Field(ge=1, le=5)
    total_score: float = Field(ge=1, le=5)
    major_losses: list[str] = Field(default_factory=list)
    hallucinations: list[str] = Field(default_factory=list)
    rationale: str


class PairQualityJudgement(BaseModel):
    set_id: int
    no_mask: ChainQualityScore
    masked: ChainQualityScore
    comparative_preference: Literal["no_mask_better", "masked_better", "tie"]
    masking_delta_summary: str
    unresolved_mask_tokens: list[str] = Field(default_factory=list)
    final_recommendation: str


JUDGE_SYSTEM_PROMPT = """
당신은 채용 분석 리포트와 면접 질문지의 생성 품질을 평가하는 엄격한 LLM judge입니다.
목표는 마스킹 체인이 마스킹 없는 체인 대비 핵심 정보 손실, 환각, 복호화 오류를 일으키는지 평가하는 것입니다.

모든 점수는 1~5점입니다.
5점: 원본 근거와 생성 체크리스트에 충실하며 채용 검토에 바로 사용 가능
4점: 사소한 누락은 있으나 핵심 판단에는 문제 없음
3점: 일부 핵심 근거가 약하거나 질문/리포트 중 하나의 품질이 불안정
2점: 중요한 내용 손실, 부정확한 일반화, 근거 없는 문장이 여러 개 있음
1점: 원본과 의미가 크게 다르거나 채용 판단에 쓰기 어려움

반드시 원본 입력과 각 체인에서 생성한 체크리스트에 있는 정보만 근거로 판단하세요.
원본에 없는 경험, 성과 수치, 회사 내부 정보, 기술 숙련도를 만들면 hallucination_control을 낮게 주세요.
마스킹 체인은 복호화된 최종 결과를 평가하되, 잔여 마스킹 토큰이나 잘못 복원된 엔티티가 있으면 entity_recovery와 total_score를 낮게 주세요.
total_score는 7개 세부 지표의 산술 평균으로 계산하세요.
모든 설명은 한국어로 작성하세요.
""".strip()


def compact_json(obj: Any, max_chars: int = 24000) -> str:
    text = json.dumps(obj, ensure_ascii=False, indent=2)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n...TRUNCATED..."


def build_judge_payload(record: dict[str, Any]) -> dict[str, Any]:
    return {
        "set_id": record["set_id"],
        "source_input": record["source"],
        "no_mask_output": {
            "generated_checklist": record["no_mask"]["checklist_generation"]["checklist"],
            "report": record["no_mask"]["report"],
            "questions": record["no_mask"]["questions"],
        },
        "masked_then_unmasked_output": {
            "mask_result": record["masked"]["mask_result"],
            "generated_checklist_after_unmask": record["masked"]["unmasked_checklist_generation"]["checklist"],
            "report": record["masked"]["report"],
            "questions": record["masked"]["questions"],
            "unresolved_mask_tokens_detected_by_regex": find_mask_tokens(record["masked"]["full_result"]),
        },
    }


def judge_pair(record: dict[str, Any]) -> dict[str, Any]:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user", "content": "다음 JSON을 평가하세요.\n\n" + compact_json(build_judge_payload(record))},
    ]
    parse_method = getattr(client.beta.chat.completions, "parse", None)
    if parse_method:
        response = parse_method(
            model=JUDGE_MODEL,
            messages=messages,
            response_format=PairQualityJudgement,
            temperature=0,
        )
        return response.choices[0].message.parsed.model_dump()

    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0,
    )
    return PairQualityJudgement.model_validate_json(response.choices[0].message.content).model_dump()

## LLM judge 실행

이미 평가 결과 캐시가 있으면 재사용합니다. 다시 평가하려면 `MASKING_QUALITY_FORCE_REJUDGE=1`을 설정하세요.

In [ ]:
if JUDGE_CACHE_PATH.exists() and not FORCE_REJUDGE:
    judge_records = load_json(JUDGE_CACHE_PATH)
    print(f"judge 캐시 로드: {JUDGE_CACHE_PATH} ({len(judge_records)}건)")
else:
    judge_records = []
    for record in chain_records:
        sid = record["set_id"]
        print(f"[set_id={sid}] LLM judge 실행")
        judgement = judge_pair(record)
        judge_records.append(judgement)
        save_json(JUDGE_CACHE_PATH, judge_records)

if pd is not None:
    display(pd.json_normalize(judge_records))
else:
    display(judge_records)

## 최종 수치 비교

아래 표의 `delta_masked_minus_no_mask`가 핵심 비교값입니다. 음수이면 마스킹 체인에서 품질 손실이 생긴 것이고, 양수이면 마스킹 체인이 더 좋은 평가를 받은 것입니다.

In [ ]:
METRICS = [
    "source_fidelity",
    "critical_information_retention",
    "coverage",
    "hallucination_control",
    "report_quality",
    "question_quality",
    "entity_recovery",
    "total_score",
]


def flatten_dict(obj: dict[str, Any], prefix: str = "") -> dict[str, Any]:
    out = {}
    for key, value in obj.items():
        path = f"{prefix}.{key}" if prefix else key
        if isinstance(value, dict):
            out.update(flatten_dict(value, path))
        else:
            out[path] = value
    return out


judge_rows = [flatten_dict(item) for item in judge_records]

summary_rows = []
for metric in METRICS:
    no_values = [float(row[f"no_mask.{metric}"]) for row in judge_rows]
    masked_values = [float(row[f"masked.{metric}"]) for row in judge_rows]
    no_avg = sum(no_values) / len(no_values)
    masked_avg = sum(masked_values) / len(masked_values)
    summary_rows.append(
        {
            "metric": metric,
            "no_mask_avg": no_avg,
            "masked_avg": masked_avg,
            "delta_masked_minus_no_mask": masked_avg - no_avg,
            "no_mask_min": min(no_values),
            "masked_min": min(masked_values),
        }
    )

pairwise_rows = []
for row in judge_rows:
    no_total = float(row["no_mask.total_score"])
    masked_total = float(row["masked.total_score"])
    pairwise_rows.append(
        {
            "set_id": row["set_id"],
            "comparative_preference": row["comparative_preference"],
            "no_mask_total": no_total,
            "masked_total": masked_total,
            "total_delta": masked_total - no_total,
            "masking_delta_summary": row["masking_delta_summary"],
            "final_recommendation": row["final_recommendation"],
        }
    )

if pd is not None:
    summary_df = pd.DataFrame(summary_rows)
    pairwise_df = pd.DataFrame(pairwise_rows)
    display(summary_df)
    display(pairwise_df)
    summary_df.to_csv(EVAL_DIR / "masking_quality_metric_summary.csv", index=False, encoding="utf-8")
    pairwise_df.to_csv(EVAL_DIR / "masking_quality_pairwise_summary.csv", index=False, encoding="utf-8")
    pd.DataFrame(judge_rows).to_csv(EVAL_DIR / "masking_quality_judge_detail.csv", index=False, encoding="utf-8")
else:
    display(summary_rows)
    display(pairwise_rows)

print("저장 파일:")
print(EVAL_DIR / "masking_quality_metric_summary.csv")
print(EVAL_DIR / "masking_quality_pairwise_summary.csv")
print(EVAL_DIR / "masking_quality_judge_detail.csv")